In [ ]:
# Setup: mount Drive, copy code + data to the fast local disk, decode images once
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, glob
# find the uploaded folder anywhere in My Drive (the one containing cifar_resnet.py + teacher_t45.pt)
hits = [os.path.dirname(p) for p in glob.glob('/content/drive/MyDrive/**/teacher_t45.pt', recursive=True)
        if os.path.exists(os.path.join(os.path.dirname(p), 'cifar_resnet.py'))]
assert hits, 'Upload not found in My Drive yet - wait for the upload to finish (all 6 files), then re-run'
DRIVE = hits[0]
print('using folder:', DRIVE, '->', sorted(os.listdir(DRIVE)))
OUT = f'{DRIVE}/runs'                                # checkpoints/results saved on Drive (survive disconnects)
os.makedirs(OUT, exist_ok=True)
os.chdir('/content')
for f in ['cifar_resnet.py', 'predict.py', 'teacher_t45.pt']:
    if not os.path.exists(f):
        shutil.copy(f'{DRIVE}/{f}', f)
if not os.path.exists('data/train_labels.csv'):
    shutil.unpack_archive(f'{DRIVE}/data.zip', 'data')
WORKERS = min(8, os.cpu_count())
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python cifar_resnet.py --name prepare --prepare_only --data_dir data --cache_dir cache --out_dir prep
# PyramidNet-110 (alpha 96) + ShakeDrop = 4,804,512 params, taught by the WRN-28-10 teacher (98.02%)
COMMON = (f'--arch pyr110_sd --epochs 300 --wd 1e-4 --teacher_ckpt teacher_t45.pt --workers {WORKERS} '
          f'--data_dir data --cache_dir cache --out_dir "{OUT}" --resume')

In [ ]:
# Run 1 (~2.5-3 h): train on 45k, score on the 5k held-back images -> tells us if PyramidNet beats our best (97.84%)
!python cifar_resnet.py --name pk45 --val_split 5000 {COMMON}

In [ ]:
# Run 2 (~2.5-3 h): same model on all 50k images -> the submission model
!python cifar_resnet.py --name pk50 {COMMON}

In [ ]:
# Predictions with 10-view TTA: held-back score for pk45, submission CSV from pk50 (saved to Drive)
!python predict.py "{OUT}/pk45/final.pt" pk45_heldout.csv --val --cache_dir cache
!python predict.py "{OUT}/pk50/final.pt" "{DRIVE}/submission_pyramidnet_pk50.csv" --cache_dir cache

In [ ]:
# Held-back accuracy over training: PyramidNet (pk45) vs our current best Wide-ResNet (kd45)
import json, matplotlib.pyplot as plt
def curve(path):
    h = [r for r in json.load(open(path))['history'] if 'val_acc_ema' in r]
    return [r['epoch'] for r in h], [100 * r['val_acc_ema'] for r in h]
pk, kd = json.load(open(f'{OUT}/pk45/metrics.json')), json.load(open(f'{DRIVE}/baseline_kd45_metrics.json'))
print('PyramidNet pk45:', pk['summary']); print('Wide-ResNet kd45:', kd['summary'])
plt.figure(figsize=(8, 4.5))
for path, label in [(f'{OUT}/pk45/metrics.json', 'PyramidNet-110 + ShakeDrop (4.80M)'),
                    (f'{DRIVE}/baseline_kd45_metrics.json', 'Wide-ResNet-28 (4.84M), round 2')]:
    plt.plot(*curve(path), label=label)
plt.ylim(90, 99); plt.xlabel('epoch'); plt.ylabel('held-back accuracy (%)'); plt.grid(alpha=.3); plt.legend()
plt.title('Held-back 5k accuracy (EMA weights), both students taught by the same teacher'); plt.show()

**Pipeline.** The student is a PyramidNet-110 (α = 96, basic blocks) with ShakeDrop: a ResNet whose width grows by a few channels at every block instead of in three jumps, with parameter-free zero-padded shortcuts, 4,804,512 trainable parameters in total. It trains from scratch for 300 epochs (SGD + Nesterov, lr 0.1 with 5-epoch warmup and cosine decay, weight decay 1e-4, batch 128, fp16) on random crop + flip + TrivialAugment + Cutout with label smoothing 0.1 and a weight EMA. It is also taught by the 36M-parameter WRN-28-10 teacher from round 1 (KD loss: 0.9·KL at T = 4 + 0.1·CE); the teacher is never submitted.

**Why.** PyramidNet gets more accuracy per parameter than Wide-ResNet (its 3.8M version matched a 36M WRN in the original paper), and ShakeDrop is the regularizer behind several CIFAR-10 records. `pk45` is scored on the same 5k held-back split as round 2. Submit `submission_pyramidnet_pk50.csv` only if `pk45` beats kd45's 97.84% (10-view TTA). If Colab disconnects, re-run all cells: `--resume` continues from the last checkpoint on Drive (saved every 10 epochs).